# 02 — PMF from real umbrella-sampling data: MBAR vs. Zwanzig

This notebook applies MBAR and a chained single-step Zwanzig estimator to a real umbrella-sampling dataset: the chi torsion of a valine sidechain in T4 lysozyme L99A with benzene bound in the cavity (Mobley et al., *J. Mol. Biol.* 371(4):1118-1134, 2007). The dataset is distributed with the `pymbar` umbrella-sampling examples.

**What this notebook does:**
1. Load the 26-window umbrella-sampling dataset biasing the chi torsion
2. Subsample each window's time series for statistical independence
3. Compute the 1D PMF with MBAR (`src/mbar_estimator.py`)
4. Compute relative window free energies with a **chained single-step Zwanzig estimator** (`src/zwanzig_estimator.py`) and compare them with MBAR
5. Estimate uncertainty in the Zwanzig chain by bootstrap resampling (`src/bootstrap.py`)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, "..")

from pymbar import timeseries
from src.umbrella_data import load_umbrella_dataset, build_reduced_potentials, flatten_u_kln
from src.mbar_estimator import MBAREstimator
from src.zwanzig_estimator import zwanzig_pairwise_chain
from src.bootstrap import bootstrap_estimator

DATA_DIR = "../data/umbrella_lysozyme_chi"

## 1. Load the raw umbrella-sampling data

In [ ]:
ds = load_umbrella_dataset(DATA_DIR)
chi0_k, Kspring_k, chi_kn_raw = ds["chi0_k"], ds["Kspring_k"], ds["chi_kn"]
K = ds["K"]

print(f"{K} umbrella windows")
print("window centers (deg):", np.round(chi0_k, 1))
print("raw samples per window:", [len(c) for c in chi_kn_raw])

## 2. Subsample for statistical independence

Each window's chi time series is correlated in time. I estimate the statistical
inefficiency `g` from `cos(chi)` and `sin(chi)` (chi is periodic, so we can't use chi itself
directly) and keep only the effectively-independent samples, following the same approach as
pymbar's own reference umbrella-sampling script for this dataset.

In [ ]:
chi_kn = []
g_k = []
for k in range(K):
    chi_rad = np.deg2rad(chi_kn_raw[k])
    g_cos = timeseries.statistical_inefficiency(np.cos(chi_rad))
    g_sin = timeseries.statistical_inefficiency(np.sin(chi_rad))
    g = max(g_cos, g_sin)
    idx = timeseries.subsample_correlated_data(chi_rad, g=g)
    chi_kn.append(chi_kn_raw[k][idx])
    g_k.append(g)

N_k_raw = np.array([len(c) for c in chi_kn_raw])
N_k_sub = np.array([len(c) for c in chi_kn])
print(f"total samples: {N_k_raw.sum()} raw -> {N_k_sub.sum()} after subsampling "
      f"({100*N_k_sub.sum()/N_k_raw.sum():.0f}% retained)")
print("statistical inefficiency g per window (min/median/max):",
      np.round([min(g_k), np.median(g_k), max(g_k)], 1))

## 3. Build reduced potentials and compute the PMF with MBAR

In [ ]:
u_kln, N_k = build_reduced_potentials(chi0_k, Kspring_k, chi_kn)
u_kn = flatten_u_kln(u_kln, N_k)

estimator = MBAREstimator(u_kn, N_k)

x_n = np.concatenate(chi_kn)
u_n = np.zeros(len(x_n))

bin_edges = np.linspace(-180, 180, 37)
pmf = estimator.compute_pmf(u_n, x_n, bin_edges)

print("PMF computed over", len(pmf["bin_centers"]), "bins")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.errorbar(pmf["bin_centers"], pmf["f_i"], yerr=pmf["df_i"], fmt="o-", ms=3, capsize=2,
            color="tab:blue", label="MBAR PMF")
ax.set_xlabel(r"Valine $\chi$ torsion (degrees)")
ax.set_ylabel(r"PMF, $f(\chi)$ (units of $k_BT$)")
ax.set_title("1D PMF of the lysozyme L99A valine chi torsion\n(umbrella sampling + MBAR)")
ax.legend()
fig.tight_layout()
fig.savefig("../results/figures/lysozyme_chi_pmf_mbar.png", dpi=150)
plt.show()

## 4. Sampling overlap between umbrella windows

MBAR is most reliable when neighboring umbrella states share configurations with appreciable statistical weight. The MBAR overlap matrix provides a direct diagnostic of this connectivity before interpreting the free-energy estimates.


In [ ]:
order = np.argsort(chi0_k)
chi0_overlap = chi0_k[order]
overlap = estimator.overlap_matrix()[np.ix_(order, order)]

fig, ax = plt.subplots(figsize=(6.2, 5.2))
im = ax.imshow(overlap, origin="lower", interpolation="nearest", aspect="auto",
              extent=[chi0_overlap[0], chi0_overlap[-1], chi0_overlap[0], chi0_overlap[-1]])
ax.set_xlabel(r"Umbrella center $\\chi_0$ (degrees)")
ax.set_ylabel(r"Umbrella center $\\chi_0$ (degrees)")
ax.set_title("MBAR state-overlap matrix")
fig.colorbar(im, ax=ax, label="Overlap")
fig.tight_layout()
fig.savefig("../results/figures/mbar_state_overlap.png", dpi=150)
plt.show()

adjacent_overlap = np.diag(overlap, k=1)
weakest_idx = np.argmin(adjacent_overlap)
print("Adjacent-window overlap (min/median/max):",
      np.round([adjacent_overlap.min(), np.median(adjacent_overlap), adjacent_overlap.max()], 3))
print("Weakest adjacent pair:",
      f"{chi0_overlap[weakest_idx]:.1f} to {chi0_overlap[weakest_idx + 1]:.1f} deg "
      f"(overlap={adjacent_overlap[weakest_idx]:.3f})")


### Interpretation

The overlap matrix shows whether neighboring umbrella windows form a connected sampling network. Stronger off-diagonal overlap means that configurations sampled in one state also carry appreciable statistical weight in another. Weak overlap identifies regions where free-energy estimates are expected to become more sensitive to sampling noise and where a chained Zwanzig calculation is particularly vulnerable to accumulated uncertainty.


## 4. Compare against a chained single-step Zwanzig estimator

MBAR pools information from all 26 windows simultaneously. For comparison, the same
relative free energies are estimated with **single-step Zwanzig perturbation** applied
as a chain across adjacent umbrella windows. Each step uses samples from the earlier
window to estimate the free-energy change to the next window, while MBAR uses all
sampled states simultaneously.

In [ ]:
order = np.argsort(chi0_k)
chi0_sorted = chi0_k[order]
Kspring_sorted = Kspring_k[order]
chi_kn_sorted = [chi_kn[i] for i in order]

u_kln_sorted, N_k_sorted = build_reduced_potentials(chi0_sorted, Kspring_sorted, chi_kn_sorted)
u_kn_sorted = flatten_u_kln(u_kln_sorted, N_k_sorted)

estimator_sorted = MBAREstimator(u_kn_sorted, N_k_sorted)
f_k_mbar, df_k_mbar = estimator_sorted.free_energies_relative_to_state_0()
f_k_zwanzig = zwanzig_pairwise_chain(u_kn_sorted, N_k_sorted)

## 5. Bootstrap error bars for the Zwanzig chain
Unlike MBAR, Zwanzig has no built-in uncertainty estimate here, so we resample each window's samples with replacement and re-run the whole chained estimate to get a standard error.

In [ ]:
def zwanzig_chain_estimator(resampled_chi_kn):
    u_kln_bs, N_k_bs = build_reduced_potentials(chi0_sorted, Kspring_sorted, resampled_chi_kn)
    u_kn_bs = flatten_u_kln(u_kln_bs, N_k_bs)
    return zwanzig_pairwise_chain(u_kn_bs, N_k_bs)

boot = bootstrap_estimator(chi_kn_sorted, zwanzig_chain_estimator, n_bootstrap=100, seed=0)
df_k_zwanzig = boot["std"]

print("Zwanzig bootstrap SE (first 8 windows):", np.round(df_k_zwanzig[:8], 2))

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.errorbar(chi0_sorted, f_k_mbar, yerr=df_k_mbar, fmt="o-", ms=3, capsize=2,
            label="MBAR (analytical SE)", color="tab:blue")
ax.errorbar(chi0_sorted, f_k_zwanzig, yerr=df_k_zwanzig, fmt="s--", ms=3, capsize=2,
            label="Zwanzig chain (bootstrap SE)", color="tab:red", alpha=0.8)
ax.set_xlabel(r"Umbrella window center, $\chi_0$ (degrees)")
ax.set_ylabel(r"$f_k - f_0$ (units of $k_BT$)")
ax.set_title("MBAR vs. chained Zwanzig:\nfree energy of each umbrella window")
ax.legend()
fig.tight_layout()
fig.savefig("../results/figures/mbar_vs_zwanzig_chain.png", dpi=150)
plt.show()

## 6. Cross-check MBAR uncertainty with bootstrap resampling

MBAR provides analytical uncertainty estimates. As a robustness check, the same per-window bootstrap procedure is used to recompute the MBAR relative free energies. Agreement between the analytical and bootstrap standard errors increases confidence that the uncertainty scale is not an artifact of a single error-estimation method.


In [ ]:
def mbar_chain_estimator(resampled_chi_kn):
    u_kln_bs, N_k_bs = build_reduced_potentials(chi0_sorted, Kspring_sorted, resampled_chi_kn)
    u_kn_bs = flatten_u_kln(u_kln_bs, N_k_bs)
    estimator_bs = MBAREstimator(u_kn_bs, N_k_bs)
    return estimator_bs.free_energies_relative_to_state_0()[0]

mbar_boot = bootstrap_estimator(
    chi_kn_sorted,
    mbar_chain_estimator,
    n_bootstrap=100,
    seed=1,
)
df_k_mbar_boot = mbar_boot["std"]

ratio = np.divide(
    df_k_mbar_boot,
    df_k_mbar,
    out=np.full_like(df_k_mbar_boot, np.nan),
    where=df_k_mbar > 0,
)
print("MBAR analytical vs bootstrap SE ratio (median):", np.nanmedian(ratio))
print("MBAR analytical vs bootstrap SE ratio (range):",
      np.round([np.nanmin(ratio), np.nanmax(ratio)], 2))


## 7. Quantitative comparison of MBAR and chained Zwanzig

To summarize the estimator comparison, the notebook reports the root-mean-square difference between the two relative free-energy profiles, the maximum absolute difference, and the typical uncertainty ratio. These metrics complement the visual comparison and make the trade-off between agreement and accumulated Zwanzig uncertainty explicit.


In [ ]:
diff_k = f_k_zwanzig - f_k_mbar
rmse = np.sqrt(np.mean(diff_k**2))
max_abs_diff = np.max(np.abs(diff_k))
median_zw_se = np.median(df_k_zwanzig)
median_mbar_se = np.median(df_k_mbar)
se_ratio = median_zw_se / median_mbar_se

print(f"Profile RMSE (Zwanzig - MBAR): {rmse:.3f} kBT")
print(f"Maximum absolute difference: {max_abs_diff:.3f} kBT")
print(f"Median Zwanzig bootstrap SE: {median_zw_se:.3f} kBT")
print(f"Median MBAR analytical SE: {median_mbar_se:.3f} kBT")
print(f"Median SE ratio (Zwanzig / MBAR): {se_ratio:.2f}")

summary = np.column_stack([
    chi0_sorted, f_k_mbar, df_k_mbar, f_k_zwanzig, df_k_zwanzig,
    df_k_mbar_boot, diff_k,
])
np.savetxt(
    "../results/mbar_vs_zwanzig_summary.csv",
    summary,
    delimiter=",",
    header="chi0_deg,mbar_free_energy,mbar_analytical_se,zwanzig_free_energy,zwanzig_bootstrap_se,mbar_bootstrap_se,zwanzig_minus_mbar",
    comments="",
)


In [ ]:
for chi, mbar, dmbar, zw, dzw in zip(
    chi0_sorted, f_k_mbar, df_k_mbar, f_k_zwanzig, df_k_zwanzig
):
    print(
        f"{chi:7.1f}  "
        f"MBAR={mbar:8.3f} ± {dmbar:6.3f}  "
        f"Zwanzig={zw:8.3f} ± {dzw:6.3f}  "
        f"diff={zw-mbar:+8.3f}"
    )

### Takeaway

MBAR and the chained Zwanzig estimates agree closely over much of the umbrella-sampling range, particularly near the reference window. The Zwanzig estimate becomes less precise as more pairwise steps are accumulated, with its bootstrap uncertainty increasing along the chain.

Larger differences appear for some windows later in the chain, reaching a few $k_BT$, while MBAR retains smaller uncertainties by using information from all sampled states simultaneously. This illustrates the main practical limitation of chaining single-step free-energy perturbation estimates: uncertainty and errors from individual steps can accumulate, whereas MBAR combines the overlapping umbrella windows in a single multistate estimate.
